# Titanic Dataset: Exploratory Data Analysis and Feature Preparation

**Objective:** Select a dataset, explore its structure, handle missing values, encode categorical variables, and prepare features for machine-learning training.

**Dataset:** Titanic passenger dataset from the Seaborn data repository. The target variable is `survived` (`0` = did not survive, `1` = survived).

**Workflow**
1. Load and inspect the dataset
2. Perform exploratory data analysis (EDA)
3. Identify and handle missing values
4. Encode categorical features and scale numerical features
5. Split data into training and test sets
6. Build a reusable preprocessing pipeline and verify its output

> This notebook focuses on data preparation. It does not claim model performance because training and evaluation are outside the requested scope.


## 1. Import libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


## 2. Load the dataset

The CSV is loaded directly from the public Seaborn data repository. An internet connection is required the first time this cell runs.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(DATA_URL)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())


## 3. Initial inspection


In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn data types:")
display(df.dtypes.to_frame("dtype"))

print("\\nSummary statistics:")
display(df.describe(include="all").T)

print("\\nDuplicate rows:", df.duplicated().sum())


## 4. Exploratory data analysis (EDA)

### 4.1 Target distribution

Checking the target distribution helps reveal whether the classes are imbalanced.


In [ ]:
target_counts = df["survived"].value_counts().sort_index()
display(target_counts.rename(index={0: "Did not survive", 1: "Survived"}).to_frame("count"))

target_counts.plot(kind="bar", rot=0)
plt.title("Survival Class Distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()


### 4.2 Missing values


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(2)
})
display(missing_summary[missing_summary["missing_count"] > 0])


### 4.3 Numerical feature distributions


In [ ]:
numeric_for_eda = [c for c in ["age", "fare", "sibsp", "parch"] if c in df.columns]
df[numeric_for_eda].hist(figsize=(10, 6), bins=20)
plt.suptitle("Distributions of Selected Numerical Features")
plt.tight_layout()
plt.show()


### 4.4 Survival patterns across selected categories


In [ ]:
for column in ["sex", "class", "embarked"]:
    if column in df.columns:
        survival_rate = df.groupby(column, dropna=False)["survived"].mean().sort_values(ascending=False)
        display(survival_rate.rename("survival_rate").to_frame().style.format("{:.1%}"))

        survival_rate.plot(kind="bar", rot=0)
        plt.title(f"Survival Rate by {column}")
        plt.ylabel("Survival rate")
        plt.tight_layout()
        plt.show()


### 4.5 Correlation among numerical variables

Correlation is a quick exploratory check for linear relationships. It does not establish causation.


In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 6))
image = ax.imshow(corr, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 5. Prepare the data for modeling

We will predict `survived`. The `alive` column directly repeats the target as text, so it is removed to prevent target leakage. Identifiers and high-cardinality text fields such as `who`, `adult_male`, `deck`, `embark_town`, and `class` are not all needed for this baseline preparation. We keep a compact, interpretable feature set.

Missing numerical values will be filled with the median. Missing categorical values will be filled with the most frequent category, and categorical columns will be one-hot encoded. Numerical columns will be standardized. These transformations are fitted only on the training split through a scikit-learn pipeline, helping avoid data leakage.


In [ ]:
target = "survived"
feature_columns = [
    "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"
]

X = df[feature_columns].copy()
y = df[target].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())


## 6. Split into training and test sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())


## 7. Missing-value imputation, encoding, and scaling


In [ ]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training matrix:", X_train_processed.shape)
print("Processed testing matrix:", X_test_processed.shape)


## 8. Verify the processed features

One-hot encoding can produce a sparse matrix. We can retrieve the transformed feature names and inspect a small sample without converting the entire dataset unnecessarily.


In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Number of transformed features:", len(feature_names))
print("Feature names:")
print(feature_names.tolist())

preview = X_train_processed[:5]
if hasattr(preview, "toarray"):
    preview = preview.toarray()

processed_preview = pd.DataFrame(preview, columns=feature_names)
display(processed_preview)


## 9. Final checks

Confirm that the processed matrices contain no missing or infinite values. The pipeline handles future unseen categorical values using `handle_unknown="ignore"`.


In [ ]:
train_array = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else np.asarray(X_train_processed)
test_array = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else np.asarray(X_test_processed)

print("Missing values in processed training data:", np.isnan(train_array).sum())
print("Missing values in processed test data:", np.isnan(test_array).sum())
print("Infinite values in processed training data:", np.isinf(train_array).sum())
print("Infinite values in processed test data:", np.isinf(test_array).sum())

assert not np.isnan(train_array).any()
assert not np.isnan(test_array).any()
assert not np.isinf(train_array).any()
assert not np.isinf(test_array).any()
print("Preprocessing checks passed.")


## 10. Conclusion

In this notebook, we:
- Selected and loaded a real-world passenger dataset.
- Inspected dimensions, data types, summary statistics, and duplicates.
- Explored the target distribution, missing values, numerical distributions, category-level survival rates, and numerical correlations.
- Separated the target variable from the input features and removed potential target leakage from the feature set.
- Split the data into training and test sets using stratification.
- Imputed missing values, standardized numerical variables, and one-hot encoded categorical variables.
- Verified that the processed feature matrices contain no missing or infinite values.

**Next step:** A classification model such as logistic regression or a decision tree can be trained using `X_train_processed` and `y_train`, then evaluated on `X_test_processed` and `y_test`.
